# 02 — SQL Exploration

Loads the raw KOI data from notebook 01 into a SQLite database, then explores
it with SQL before any pandas analysis — the way you'd interrogate a real
database in a data job.

In [24]:
import pandas as pd
import sqlite3
from pathlib import Path

In [25]:
# Find columns containing integers too big for SQLite's 64-bit limit (~9.2e18)
SQLITE_MAX = 9_223_372_036_854_775_807

for col in df.columns:
    # only check numeric columns
    if pd.api.types.is_numeric_dtype(df[col]):
        col_max = df[col].abs().max()
        if pd.notna(col_max) and col_max > SQLITE_MAX:
            print(f"{col}: max abs value = {col_max}")

In [26]:
df = pd.read_csv("../data/raw/koi_cumulative.csv")

conn = sqlite3.connect("../data/koi.db")

# Convert to strings before loading: some catalog values exceed SQLite's
# 64-bit integer limit and overflow at the pandas→sqlite handoff. Storing
# as TEXT sidesteps it; we CAST back to numbers inside queries that do math.
df_text = df.astype(str)
df_text.to_sql("koi", conn, if_exists="replace", index=False)

print("Loaded", len(df_text), "rows into table 'koi'")

Loaded 9564 rows into table 'koi'


In [27]:
def q(sql):
    """Run a SQL query against the koi database, return results as a DataFrame."""
    return pd.read_sql_query(sql, conn)

In [28]:
q("""
SELECT koi_disposition, COUNT(*) AS n
FROM koi
GROUP BY koi_disposition
""")

,koi_disposition,n
0,CANDIDATE,1978
1,CONFIRMED,2747
2,FALSE POSITIVE,4839


**Interpretation:** The three classes are imbalanced. False positives are the
largest group at 4,839 (~51% of all rows), followed by 2,747 confirmed planets
and 1,978 candidates. Because roughly half the data is one class, a model that
blindly guessed "false positive" every time would already be ~51% accurate —
which is why accuracy alone will be misleading later, and why the modeling
notebook will lean on precision, recall, and F1 instead.

In [29]:
q("""
SELECT koi_disposition,
       ROUND(AVG(CAST(koi_period AS REAL)), 2) AS avg_period,
       ROUND(AVG(CAST(koi_prad AS REAL)), 2) AS avg_radius
FROM koi
GROUP BY koi_disposition
""")

,koi_disposition,avg_period,avg_radius
0,CANDIDATE,167.78,97.95
1,CONFIRMED,27.90,2.86
2,FALSE POSITIVE,65.14,164.84


**Interpretation:** Average planet radius separates the classes sharply.
Confirmed planets average just 2.86 Earth radii, while false positives average
164.84 — roughly 57× larger. That gap makes physical sense: many false
positives are eclipsing binaries (two stars), and a star dwarfs a planet, so an
implausibly huge "planet" radius is a strong signal the object isn't a planet at
all. This suggests `koi_prad` will be a useful predictive feature later.

Orbital period shows no clean pattern across classes, so it looks less
discriminating on its own.

Caveat: these are averages, which outliers can distort — the false-positive
radius mean is likely inflated by a few extreme values. The median would be more
robust, and outlier handling is a job for the cleaning notebook.

In [30]:
q("""
SELECT COUNT(*) AS missing_radius
FROM koi
WHERE koi_prad IS NULL OR koi_prad != koi_prad
""")

,missing_radius
0,363


**Interpretation:** `koi_prad` has **363 missing values** — but they only
surfaced after probing how missingness was encoded. An initial check for the
string `'nan'` returned 0, because the values are stored as true numeric NaN,
not text. NaN's defining quirk — it never equals itself — is what finally
caught them (`koi_prad != koi_prad`). Lesson: a "0 missing" result is worth a
second look, since missing data hides under many encodings. Handling these 363
gaps is a job for the cleaning notebook.